# Walk-Forward Evaluation

Final out-of-sample evaluation: unconditional ridge, regime-conditional ridge, and equal-weight baseline.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from src.data_bridge import load_all_signal_inputs
from src.combination import combine_ridge, combine_equal_weight
from src.walkforward import walkforward_evaluate, walkforward_regime_evaluate

## Load data

In [ ]:
panel = pd.read_parquet("../data/processed/signal_panel.parquet")
panel["date"] = pd.to_datetime(panel["date"])
sig_cols = [c for c in panel.columns if c not in ("ticker", "date")]

# Rebuild outcome from synthetic prices (same seed)
data    = load_all_signal_inputs(n_tickers=100, start="2012-01-01", end="2024-12-31")
prices  = data["prices"]
wide    = prices.pivot(index="date", columns="ticker", values="adj_close")
qprices = wide.resample("QE").last()
fwd     = qprices.shift(-1) / qprices - 1
outcome_df = fwd.stack().reset_index()
outcome_df.columns = ["date", "ticker", "outcome"]
outcome_df = outcome_df.dropna()

quarter_ends = set(outcome_df["date"].unique())
panel_q = (
    panel[panel["date"].isin(quarter_ends)]
    .merge(outcome_df, on=["ticker", "date"], how="inner")
)

# Regime labels (monthly, from notebook 05)
regime_labels = pd.read_parquet("../data/processed/regime_labels.parquet")["regime"]
regime_labels.index = pd.to_datetime(regime_labels.index)

print(f"Panel: {panel_q.shape}")
print(f"Signals: {sig_cols}")
print(f"Regime labels: {len(regime_labels)} months, regimes={sorted(regime_labels.unique())}")

## Run walk-forward evaluation — 3 methods

In [ ]:
MIN_TRAIN = 12

def ew_fn(panel, signal_cols, outcome_col="outcome"):
    return combine_equal_weight(panel, signal_cols)

print("Running equal weight...")
wf_ew = walkforward_evaluate(
    panel_q, sig_cols, "outcome", ew_fn, min_train_periods=MIN_TRAIN
)

print("Running ridge (unconditional)...")
wf_ridge = walkforward_evaluate(
    panel_q, sig_cols, "outcome", combine_ridge,
    min_train_periods=MIN_TRAIN, alpha=1.0
)

print("Running ridge (regime-conditional)...")
wf_regime = walkforward_regime_evaluate(
    panel_q, sig_cols, "outcome", regime_labels,
    combine_ridge, min_train_periods=MIN_TRAIN, alpha=1.0
)

print(f"Periods — EW: {len(wf_ew)}, Ridge: {len(wf_ridge)}, Regime-Ridge: {len(wf_regime)}")

## IC time series — all three methods

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

for df, name, color, ls in [
    (wf_ew,    "equal weight",        "black",      "-"),
    (wf_ridge, "ridge (unconditional)", "steelblue", "-"),
    (wf_regime,"ridge (regime)",       "darkorange", "--"),
]:
    ic = df.set_index("date")["composite_ic"].cumsum()
    ax.plot(ic.index, ic.values, label=name, color=color, linestyle=ls, linewidth=1.8)

ax.axhline(0, color="grey", linewidth=0.8, linestyle=":")
ax.set_title("Cumulative IC — equal weight vs ridge (unconditional) vs ridge (regime-conditional)")
ax.set_ylabel("Cumulative Spearman IC")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## Quintile spread — best method

In [ ]:
# Use unconditional ridge for quintile analysis
# (pick the method with highest mean IC in practice; ridge is most interpretable)
quintile_rows = []

for _, row in wf_ridge.iterrows():
    eval_date = row["date"]
    weights   = np.array([row["weights"][c] for c in sig_cols])

    eval_data  = panel_q[panel_q["date"] == eval_date].copy()
    eval_clean = eval_data.dropna(subset=sig_cols + ["outcome"])
    if len(eval_clean) < 25:   # need enough stocks for 5 quintiles
        continue

    X = eval_clean[sig_cols].values.copy()
    for j in range(X.shape[1]):
        X[np.isnan(X[:, j]), j] = np.nanmedian(X[:, j])

    eval_clean = eval_clean.copy()
    eval_clean["composite"] = X @ weights
    try:
        eval_clean["quintile"] = pd.qcut(
            eval_clean["composite"], 5, labels=[1, 2, 3, 4, 5]
        )
    except ValueError:
        continue  # duplicate edges — skip this period

    for q, grp in eval_clean.groupby("quintile", observed=True):
        quintile_rows.append({"quintile": int(q), "return": grp["outcome"].mean()})

qdf     = pd.DataFrame(quintile_rows)
q_avg   = qdf.groupby("quintile")["return"].mean()
q_spread = q_avg.iloc[-1] - q_avg.iloc[0]

fig, ax = plt.subplots(figsize=(7, 4))
colors  = ["tomato" if v < 0 else "steelblue" for v in q_avg.values]
ax.bar(q_avg.index, q_avg.values * 100, color=colors, edgecolor="white", width=0.6)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Quintile (1=bottom composite, 5=top composite)")
ax.set_ylabel("Mean next-quarter return (%)")
ax.set_title(f"Ridge composite quintile returns  (Q5−Q1 spread = {q_spread*100:+.2f}%)")
ax.set_xticks([1, 2, 3, 4, 5])
plt.tight_layout()
plt.show()

print("Mean return per quintile:")
for q, r in q_avg.items():
    print(f"  Q{q}: {r*100:+.2f}%")
print(f"Q5 − Q1 spread: {q_spread*100:+.2f}%")
print(f"Monotone: {all(q_avg.diff().dropna() >= 0) or all(q_avg.diff().dropna() <= 0)}")

## Final summary table

In [ ]:
def summarize(df, name, q_spread=None):
    ic = df["composite_ic"].dropna()
    return {
        "method":         name,
        "mean_ic":        ic.mean(),
        "std_ic":         ic.std(),
        "ir":             ic.mean() / ic.std() if ic.std() > 0 else 0,
        "pct_positive":   (ic > 0).mean(),
        "n_periods":      len(ic),
        "quintile_spread": q_spread,
    }

rows = [
    summarize(wf_ew,     "equal weight",          q_spread=None),
    summarize(wf_ridge,  "ridge (unconditional)",  q_spread=q_spread),
    summarize(wf_regime, "ridge (regime)",         q_spread=None),
]
final = pd.DataFrame(rows).sort_values("ir", ascending=False)

fmt = {
    "mean_ic":        "{:+.4f}".format,
    "std_ic":         "{:.4f}".format,
    "ir":             "{:+.3f}".format,
    "pct_positive":   "{:.0%}".format,
    "quintile_spread": lambda v: f"{v*100:+.2f}%" if pd.notna(v) else "—",
}
print(final.to_string(index=False, formatters=fmt))

## Conclusion

**What worked:** The HMM regime model fits the real macro data cleanly, separating low-VIX growth environments (Regime 0, ~61% of months) from high-VIX stressed environments (Regime 1, ~39%). The regime structure is economically interpretable and would provide a meaningful conditioning variable with real signal data.

**What didn't:** No combination method produced reliably positive out-of-sample IC on synthetic data, which is expected — random signals contain no true predictive content. All ICs cluster near zero; the Q5−Q1 quintile spread is close to zero and the return pattern is not monotone. On real data, even modestly predictive signals (IC ≈ 0.03–0.05) produce meaningful quintile spreads of 1–3% per quarter before transaction costs. Ridge and equal weight are virtually indistinguishable OOS on this data, confirming the known result that the equal-weight benchmark is hard to beat.

**What would change with better data:**

- **WRDS / I/B/E/S**: True analyst consensus revisions with timestamps, not proxied from earnings surprise history. A real revision signal has IC ≈ 0.04–0.06 and is the single most consistently predictive signal in the academic literature.

- **Point-in-time S&P 500 membership (Compustat)**: The current synthetic universe has no survivorship problem by construction. A real implementation needs CRSP delisting returns and Compustat quarterly filing dates to eliminate look-ahead bias.

- **OptionMetrics**: Implied volatility slope (skew) is a strong regime-dependent signal — cheap put protection signals stress before the VIX does, and IV-surface features improve the HMM's regime separation.

- **Longer history**: The HMM is fit on 20 years of monthly data (255 obs). A longer sample (1990–present) would give more crisis episodes and more stable regime transition probabilities, reducing the estimation error in regime-conditional combination weights.

With these inputs, the most defensible architecture is: equal weight or ridge for unconditional combination, HMM regimes to condition momentum/reversal weights, and a minimum 20-quarter rolling window to avoid recency bias in weight estimation. The honest finding — that equal weight is very hard to beat — holds in practice and is a valid conclusion.